# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Соловей  
**Група:** Соловей_______  
**Дата:** Соловей________



**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

In [ ]:
# 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

print('Користувачів:', ratings_df.user_id.nunique())
print('Фільмів:', ratings_df.item_id.nunique())
print(ratings_df.rating.agg(['min','max','mean']))
print('Пропущені значення:')
print(ratings_df.isna().sum())
print('Розподіл оцінок:')
display(ratings_df.rating.value_counts().sort_index())

**Короткі спостереження щодо даних:** MovieLens 100K містить 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки належать шкалі від 1 до 5, середня оцінка становить приблизно 3,53. Найчастіше зустрічається оцінка 4.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
# 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

train_df, test_df = sklearn_train_test_split(ratings_df, test_size=0.2, random_state=42)
print('Train:', train_df.shape, 'Test:', test_df.shape)

In [ ]:
# 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

reader = Reader(rating_scale=data.reader.rating_scale)
cols = ['user_id','item_id','rating']
train_data = Dataset.load_from_df(train_df[cols], reader)
testset = list(test_df[cols].itertuples(index=False, name=None))
print('Train:', len(train_data.raw_ratings), 'Test:', len(testset))

## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:** Залишено початкову сітку викладача. Для SVD кількість факторів визначає складність моделі, швидкість навчання впливає на збіжність, а регуляризація обмежує перенавчання. Для KNNBasic параметри k і min_k визначають кількість сусідів, а cosine та pearson дозволяють порівняти різні способи оцінки схожості користувачів або фільмів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
# 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

from surprise.model_selection import KFold
cv = KFold(n_splits=3, random_state=42, shuffle=True)
param_grid_svd['random_state'] = [42]
grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse','mae'], cv=cv, n_jobs=2)
grid_search_svd.fit(train_data)

In [ ]:
# 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse','mae'], cv=cv, n_jobs=2, joblib_verbose=0)
grid_search_knn.fit(train_data)

In [ ]:
# 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

cv_rows = []
for name, search in [('SVD', grid_search_svd), ('KNNBasic', grid_search_knn)]:
    i = search.best_index['rmse']
    cv_rows.append({'Model': name, 'CV RMSE': search.best_score['rmse'],
                    'CV MAE': search.cv_results['mean_test_mae'][i],
                    'Best CV MAE': search.best_score['mae'],
                    'Parameters': search.best_params['rmse']})
cv_results = pd.DataFrame(cv_rows).set_index('Model')
display(cv_results)
print('Найкраща модель за CV RMSE:', cv_results['CV RMSE'].idxmin())

**Аналіз результатів кросвалідації:** Таблиця вище містить найкращий CV RMSE, відповідний MAE для тих самих параметрів та окремий мінімум MAE. Модель обирається за мінімальним CV RMSE; параметри наведено у стовпці Parameters.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# 6.2 — Створіть trainset із train_data.
# 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_svd_params['random_state'] = 42
best_knn_params = grid_search_knn.best_params['rmse'].copy()
trainset = train_data.build_full_trainset()
best_svd = SVD(**best_svd_params)
best_knn = KNNBasic(**best_knn_params, verbose=False)
best_svd.fit(trainset)
best_knn.fit(trainset)
print('Моделі навчені:', trainset.n_ratings)

## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# 7.1 — Отримайте svd_predictions та knn_predictions.
# 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [ ]:
# 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

results = pd.DataFrame({'Model':['SVD','KNNBasic'], 'RMSE':[svd_rmse,knn_rmse], 'MAE':[svd_mae,knn_mae]}).sort_values('RMSE')
display(results.round(4))
import matplotlib.pyplot as plt
results.set_index('Model').plot.bar(rot=0, figsize=(7,4))
plt.title('Порівняння моделей на тестовій вибірці')
plt.ylabel('Помилка')
plt.tight_layout()
plt.show()

**Таблиця тестових результатів:** наведена вище. Менші RMSE та MAE означають точніші прогнози. Числове порівняння з CV наведено у підсумковій комірці.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Початковий ID можна змінити
user_id = '196'

# 8.1 — Визначте найкращу за CV RMSE модель.
# 8.2 — Перевірте, що користувач присутній у trainset.
# 8.3 — Знайдіть неоцінені ним фільми.
# 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

winner = cv_results['CV RMSE'].idxmin()
model = best_svd if winner == 'SVD' else best_knn
try:
    inner_uid = trainset.to_inner_uid(user_id)
except ValueError:
    print('Користувач відсутній у навчальній вибірці.')
else:
    rated = {iid for iid, rating in trainset.ur[inner_uid]}
    candidates = [trainset.to_raw_iid(iid) for iid in trainset.all_items() if iid not in rated]
    top10 = pd.DataFrame([(iid, model.predict(uid=user_id, iid=iid).est) for iid in candidates], columns=['item_id','predicted_rating']).sort_values(['predicted_rating','item_id'], ascending=[False,True]).head(10).reset_index(drop=True)
    print('Модель:', winner, 'Користувач:', user_id)
    display(top10)
    assert not set(top10.item_id) & {trainset.to_raw_iid(i) for i in rated}

## 9. Перевірка виконання

Перелік перевірок після запуску ноутбука:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Числовий висновок формується автоматично після виконання моделей.

In [ ]:
print(f'За кросвалідацією обрано {winner}, оскільки ця модель має найменший RMSE ({cv_results.loc[winner, "CV RMSE"]:.4f}).')
print(f'Найкращі параметри SVD: {best_svd_params}.')
print(f'Найкращі параметри KNNBasic: {best_knn_params}.')
for row in results.itertuples(index=False):
    diff = row.RMSE - cv_results.loc[row.Model, 'CV RMSE']
    print(f'{row.Model}: тестовий RMSE = {row.RMSE:.4f}, MAE = {row.MAE:.4f}; різниця RMSE між тестом і CV = {diff:+.4f}.')
print('SVD використовує приховані фактори, тоді як KNNBasic спирається на схожість сусідів.')
print('Рекомендації обмежені наявними оцінками; для нових користувачів і фільмів виникає проблема холодного старту.')

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

### Власний алгоритм: зважене середнє оцінок
Прогнозуємо рейтинг за середньою оцінкою фільму зі згладжуванням до загального середнього та поправкою на середню оцінку користувача. Використовуємо тільки навчальні дані.

In [ ]:
global_mean = train_df.rating.mean()
item_stats = train_df.groupby('item_id').rating.agg(['sum','count'])
user_stats = train_df.groupby('user_id').rating.agg(['sum','count'])

def own_predict(uid, iid, m=10):
    item_mean = global_mean
    user_bias = 0.0
    if iid in item_stats.index:
        s = item_stats.loc[iid]
        item_mean = (s['sum'] + m * global_mean) / (s['count'] + m)
    if uid in user_stats.index:
        s = user_stats.loc[uid]
        user_bias = (s['sum'] - s['count'] * global_mean) / (s['count'] + m)
    return float(np.clip(item_mean + user_bias, 1, 5))

own_pred = np.array([own_predict(u,i) for u,i,r in testset])
actual = np.array([r for u,i,r in testset])
print('RMSE:', np.sqrt(np.mean((actual-own_pred)**2)))
print('MAE:', np.mean(np.abs(actual-own_pred)))
seen = set(train_df.loc[train_df.user_id == user_id,'item_id'])
own_top10 = pd.DataFrame([(i,own_predict(user_id,i)) for i in item_stats.index if i not in seen], columns=['item_id','predicted_rating']).sort_values(['predicted_rating','item_id'],ascending=[False,True]).head(10)
display(own_top10.reset_index(drop=True))

**Висновок до додаткового завдання:** власний алгоритм не потребує бібліотеки Surprise для прогнозування. Він враховує популярність фільму та схильність користувача ставити вищі або нижчі оцінки, але не моделює індивідуальні жанрові вподобання.